In [0]:
# =========================================================
# 03_silver
# Camada SILVER — Data Quality
# =========================================================

from pyspark.sql.functions import col, upper

# ---------------------------------------------------------
# 1. Criar schema SILVER e limpar tabelas antigas
# ---------------------------------------------------------

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

spark.sql("DROP TABLE IF EXISTS workspace.silver.clientes")
spark.sql("DROP TABLE IF EXISTS workspace.silver.produtos")
spark.sql("DROP TABLE IF EXISTS workspace.silver.vendas")


# =========================================================
# 2. CLIENTES
# =========================================================

df_clientes = spark.read.table("workspace.bronze.clientes")

df_clientes = df_clientes.dropDuplicates()
df_clientes = df_clientes.filter(col("id_cliente").isNotNull())
df_clientes = df_clientes.withColumn("nome", upper(col("nome")))
df_clientes = df_clientes.na.fill({
    "cidade": "NÃO INFORMADO",
    "estado": "NÃO INFORMADO"
})

df_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.clientes")

print(f"✅ Silver clientes: {df_clientes.count()} registros")


# =========================================================
# 3. PRODUTOS
# =========================================================

df_produtos = spark.read.table("workspace.bronze.produtos")

df_produtos = df_produtos.dropDuplicates()
df_produtos = df_produtos.filter(col("id_produto").isNotNull())
df_produtos = df_produtos.filter(col("preco") > 0)
df_produtos = df_produtos.withColumn("nome", upper(col("nome")))
df_produtos = df_produtos.na.fill({"categoria": "NÃO INFORMADO"})

df_produtos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.produtos")

print(f"✅ Silver produtos: {df_produtos.count()} registros")


# =========================================================
# 4. VENDAS
# =========================================================

df_vendas = spark.read.table("workspace.bronze.vendas")

df_vendas = df_vendas.dropDuplicates()
df_vendas = df_vendas.filter(col("id_venda").isNotNull())
df_vendas = df_vendas.filter(col("quantidade") > 0)
df_vendas = df_vendas.filter(col("valor_unitario") > 0)

df_vendas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.vendas")

print(f"✅ Silver vendas: {df_vendas.count()} registros")


# =========================================================
# 5. Validação
# =========================================================

print("\n✅ Tabelas SILVER criadas com sucesso!")
spark.sql("SHOW TABLES IN workspace.silver").show(truncate=False)